## KZSU Zookeeper API Playlist Consolidator

This notebook consolidates all playlists for a given DJ_ID from the KZSU Zookeeper API into a single CSV file conforming to zookeeper CSV upload requirements. Each track is enriched with metadata such as date, time, show name, and other relevant information.

In [1]:
import os
import re
from datetime import date, timedelta

import requests
import pandas as pd

# API configuration
KZSU_API_KEY = os.getenv('KZSU_LIBRARY_API_KEY', 'b3fcd72414bde44530b5fd7b585aaafc08ca41b1')
HEADERS = {
    'X-APIKEY': KZSU_API_KEY,
    'Accept': 'application/vnd.api+json',
    'Content-Type': 'application/vnd.api+json',
}
BASE_URL_V2 = 'https://zookeeper.stanford.edu/api/v2/'

# User configuration
DJ_NAME = 'DJ Stace'  # Use the textual airname from the show page.
SEARCH_DATES = ['2026-08-20']  # Add the dates you want to search; keep this short to avoid API throttling.
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Resolve the actual playlist IDs for a DJ by searching the specific dates you care about.
def find_playlist_ids_for_dj(dj_name, dates=None):
    dj_name = dj_name.strip().lower()
    if dates is None:
        today = date.today()
        dates = [(today - timedelta(days=offset)).isoformat() for offset in range(7)]

    playlist_ids = []
    for target_date in dates:
        response = requests.get(
            f'{BASE_URL_V2}playlist',
            headers=HEADERS,
            params={'filter[date]': target_date, 'page[size]': 100},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        data = payload.get('data', [])
        if isinstance(data, dict):
            data = [data]

        for playlist in data:
            attributes = playlist.get('attributes', {})
            if (attributes.get('airname') or '').strip().lower() == dj_name:
                playlist_ids.append(playlist['id'])

    return list(dict.fromkeys(playlist_ids))


# Function to fetch playlist events for a given playlist ID.
def fetch_playlist_events(playlist_id):
    url = f'{BASE_URL_V2}playlist/{playlist_id}/events'
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()
    data = payload.get('data', [])
    if isinstance(data, dict):
        return [data]
    return data


# Function to consolidate playlists into a single CSV.
    playlist_ids = find_playlist_ids_for_dj(dj_name, dates=SEARCH_DATES)
    playlist_ids = find_playlist_ids_for_dj(dj_name, lookback_days=LOOKBACK_DAYS)
    all_tracks = []

    if not playlist_ids:
        print(f'No playlist IDs found for: {dj_name}. Check the airname spelling or widen LOOKBACK_DAYS.')
        return

    for playlist_id in playlist_ids:
        playlist_response = requests.get(
            f'{BASE_URL_V2}playlist/{playlist_id}',
            headers=HEADERS,
            timeout=30,
        )
        playlist_response.raise_for_status()
        playlist = playlist_response.json().get('data', {})
        attributes = playlist.get('attributes', {})
        show_name = attributes.get('airname') or attributes.get('name') or ''
        date_value = attributes.get('date', '')
        time_value = attributes.get('time', '')

        for event in fetch_playlist_events(playlist_id):
            event_attrs = event.get('attributes', {})
            if event_attrs.get('type') != 'spin':
                continue

            all_tracks.append({
                'date': date_value,
                'time': time_value,
                'show_name': show_name,
                'dj_name': dj_name,
                'artist': event_attrs.get('artist', ''),
                'title': event_attrs.get('track', '') or event_attrs.get('title', ''),
                'album': event_attrs.get('album', ''),
                'label': event_attrs.get('label', ''),
                'duration': event_attrs.get('duration', ''),
            })

    df = pd.DataFrame(all_tracks)
    safe_name = re.sub(r'[^a-z0-9]+', '_', dj_name.lower()).strip('_') or 'dj'
    output_path = os.path.join(OUTPUT_DIR, f'{safe_name}_consolidated_playlists.csv')
    df.to_csv(output_path, index=False)
    print(f'Consolidated playlist saved to {output_path} ({len(df)} tracks)')


# Run consolidation
consolidate_playlists_for_dj(DJ_NAME)

NameError: name 'consolidate_playlists_for_dj' is not defined